In [1]:
import os
import sys
import traceback

import joblib
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

In [2]:
# rutas
workspace_dir  = os.getcwd()
mlruns_dir     = os.path.join(workspace_dir, "mlruns")
tracking_uri   = "file:///" + os.path.abspath(mlruns_dir).replace("\\", "/")
artifact_loc   = tracking_uri          # experimentos y modelos en el mismo directorio
model_pkl_path = os.path.join(workspace_dir, "model.pkl")

print(f"[train] CWD            : {workspace_dir}")
print(f"[train] MLRuns dir     : {mlruns_dir}")
print(f"[train] Tracking URI   : {tracking_uri}")

os.makedirs(mlruns_dir, exist_ok=True)

[train] CWD            : C:\Users\jmanu\Documents\MLops\taller4
[train] MLRuns dir     : C:\Users\jmanu\Documents\MLops\taller4\mlruns
[train] Tracking URI   : file:///C:/Users/jmanu/Documents/MLops/taller4/mlruns


In [3]:
# Configurar el mflow
mlflow.set_tracking_uri(tracking_uri)

experiment_name = "CI-CD-Lab-MLflow"
try:
    experiment_id = mlflow.create_experiment(
        name=experiment_name,
        artifact_location=artifact_loc,
    )
    print(f"[train] Experimento creado  → ID: {experiment_id}")
except mlflow.exceptions.MlflowException as exc:
    if "RESOURCE_ALREADY_EXISTS" not in str(exc):
        raise
    exp = mlflow.get_experiment_by_name(experiment_name)
    experiment_id = exp.experiment_id
    print(f"[train] Experimento existente → ID: {experiment_id}")

[train] Experimento creado  → ID: 406430150387362155


C:\Users\jmanu\AppData\Local\Programs\Python\Python311\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


In [4]:
X, y = load_diabetes(return_X_y=True, as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)
preds = model.predict(X_test)
mse   = mean_squared_error(y_test, preds)

print(f"[train] MSE en test set: {mse:.4f}")

[train] MSE en test set: 2900.1936


In [5]:
try:
    with mlflow.start_run(experiment_id=experiment_id) as run:
        # Parámetros del modelo
        mlflow.log_param("model_type", "LinearRegression")
        mlflow.log_param("test_size",  0.2)
        mlflow.log_param("random_state", 42)

        # Métricas
        mlflow.log_metric("mse", mse)

        # Artefacto: modelo con firma
        signature = infer_signature(X_train, model.predict(X_train))
        mlflow.sklearn.log_model(
            sk_model=model,
            artifact_path="model",
            signature=signature,
        )

        print(f"[train] Run ID        : {run.info.run_id}")
        print(f"[train] Artifact URI  : {run.info.artifact_uri}")

except Exception:
    traceback.print_exc()
    sys.exit(1)

2026/05/02 16:04:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/02 16:04:19 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


[train] Run ID        : 9be0ebc6d3cb47fda1c7828d13210dd1
[train] Artifact URI  : file:///C:/Users/jmanu/Documents/MLops/taller4/mlruns/9be0ebc6d3cb47fda1c7828d13210dd1/artifacts


In [6]:
joblib.dump(model, model_pkl_path)
print(f"[train] model.pkl guardado en: {model_pkl_path}")
print("[train] ✅ Entrenamiento completado exitosamente.")

[train] model.pkl guardado en: C:\Users\jmanu\Documents\MLops\taller4\model.pkl
[train] ✅ Entrenamiento completado exitosamente.
